In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import RobustScaler
from scipy.stats import skew

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

In [ ]:
df_train = pd.read_csv(r"D:\Data Analytics\Kaggle\train.csv")
x_test   = pd.read_csv(r"D:\Data Analytics\Kaggle\test.csv")
test_ids = x_test["Id"]
print(df_train.shape, x_test.shape)

In [ ]:
for df in [df_train, x_test]:
    df["Age"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"].fillna(0) + df["2ndFlrSF"].fillna(0)
    df["Quality_SF"] = df["OverallQual"] * df["TotalSF"]
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

In [ ]:
mask = ~((df_train["GrLivArea"] > 4000) & (df_train["SalePrice"] < 300000))
print("dropping", (~mask).sum())
df_train = df_train[mask].reset_index(drop=True)

In [ ]:
y_train = np.log1p(df_train["SalePrice"])
x_train = df_train.drop(columns=["SalePrice", "Id"])
x_test  = x_test.drop(columns=["Id"])
x_test  = x_test[x_train.columns]

In [ ]:
num_cols = x_train.select_dtypes(include="number").columns
cat_cols = x_train.select_dtypes(exclude="number").columns

for col in num_cols:
    m = x_train[col].median()
    x_train[col] = x_train[col].fillna(m)
    x_test[col]  = x_test[col].fillna(m)

for col in cat_cols:
    x_train[col] = x_train[col].astype(object).fillna("None").astype(str)
    x_test[col]  = x_test[col].astype(object).fillna("None").astype(str)

print("NaNs left:", x_train.isna().sum().sum(), x_test.isna().sum().sum())

In [ ]:
cat_cols = list(cat_cols)
for col in cat_cols:
    cats = sorted(set(x_train[col]) | set(x_test[col]))
    dtype = pd.CategoricalDtype(categories=cats)
    x_train[col] = x_train[col].astype(dtype)
    x_test[col]  = x_test[col].astype(dtype)

In [ ]:
all_x = pd.concat([x_train, x_test])

skewed = [c for c in num_cols if abs(skew(all_x[c])) > 0.75 and (all_x[c] >= 0).all()]
all_x[skewed] = np.log1p(all_x[skewed])

all_x = pd.get_dummies(all_x, columns=cat_cols, dtype=float)
all_x = pd.DataFrame(RobustScaler().fit_transform(all_x), columns=all_x.columns)

x_train_lin = all_x.iloc[:len(x_train)]
x_test_lin  = all_x.iloc[len(x_train):]

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(x_train))
oof_lgb = np.zeros(len(x_train))
oof_cb  = np.zeros(len(x_train))

for fold, (train_idx, val_idx) in enumerate(kf.split(x_train), 1):
    print(f"\n========== FOLD {fold} ==========")

    X_train = x_train.iloc[train_idx].copy()
    X_val   = x_train.iloc[val_idx].copy()
    Y_train = y_train.iloc[train_idx]
    Y_val   = y_train.iloc[val_idx]

    # XGBoost
    model_xgb = xgb.XGBRegressor(
        enable_categorical=True,
        tree_method="hist",
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    model_xgb.fit(X_train, Y_train)
    p_xgb = model_xgb.predict(X_val)
    oof_xgb[val_idx] = p_xgb

    # LightGBM
    X_train_lgb = X_train.copy()
    X_val_lgb   = X_val.copy()
    for col in cat_cols:
        X_train_lgb[col] = X_train_lgb[col].astype("category")
        X_val_lgb[col]   = X_val_lgb[col].astype("category")
        X_train_lgb[col] = X_train_lgb[col].cat.add_categories(["Missing"]).fillna("Missing")
        X_val_lgb[col]   = X_val_lgb[col].cat.add_categories(["Missing"]).fillna("Missing")

    model_lgb = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=4,
        num_leaves=16,
        random_state=42,
        verbose=-1
    )
    model_lgb.fit(X_train_lgb, Y_train)
    p_lgb = model_lgb.predict(X_val_lgb)
    oof_lgb[val_idx] = p_lgb

    # CatBoost
    model_cb = cb.CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=5,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )
    model_cb.fit(X_train, Y_train, cat_features=cat_cols)
    p_cb = model_cb.predict(X_val)
    oof_cb[val_idx] = p_cb

    print("XGB:", round(np.sqrt(mean_squared_error(Y_val, p_xgb)), 5))
    print("LGB:", round(np.sqrt(mean_squared_error(Y_val, p_lgb)), 5))
    print("CAT:", round(np.sqrt(mean_squared_error(Y_val, p_cb)), 5))

print("\n========== OVERALL 5-FOLD CV ==========")
xgb_score = np.sqrt(mean_squared_error(y_train, oof_xgb))
lgb_score = np.sqrt(mean_squared_error(y_train, oof_lgb))
cb_score  = np.sqrt(mean_squared_error(y_train, oof_cb))
print("XGB:", round(xgb_score, 5))
print("LGB:", round(lgb_score, 5))
print("CAT:", round(cb_score, 5))

In [ ]:
lasso = Lasso(alpha=0.0005, max_iter=50000)
oof_lasso = cross_val_predict(lasso, x_train_lin, y_train, cv=kf)
print("Lasso:", round(np.sqrt(mean_squared_error(y_train, oof_lasso)), 5))

In [ ]:
meta_X = np.column_stack([
    oof_xgb,
    oof_lgb,
    oof_cb,
    oof_lasso
])

In [ ]:
meta_model = Ridge(alpha=1.0)
stack_oof = cross_val_predict(meta_model, meta_X, y_train, cv=5)
stack_score = np.sqrt(mean_squared_error(y_train, stack_oof))
print("Stacking:", round(stack_score, 5))
meta_model.fit(meta_X, y_train)

In [ ]:
model_xgb.fit(x_train, y_train)

X_lgb = x_train.copy()
for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype("category")
    X_lgb[col] = X_lgb[col].cat.add_categories(["Missing"]).fillna("Missing")
model_lgb.fit(X_lgb, y_train)

model_cb.fit(x_train, y_train, cat_features=cat_cols)
lasso.fit(x_train_lin, y_train)

test_pred_xgb = model_xgb.predict(x_test)

X_test_lgb = x_test.copy()
for col in cat_cols:
    X_test_lgb[col] = X_test_lgb[col].astype("category")
    X_test_lgb[col] = X_test_lgb[col].cat.add_categories(["Missing"]).fillna("Missing")
test_pred_lgb = model_lgb.predict(X_test_lgb)

test_pred_cb    = model_cb.predict(x_test)
test_pred_lasso = lasso.predict(x_test_lin)

meta_test_X = np.column_stack([test_pred_xgb, test_pred_lgb, test_pred_cb, test_pred_lasso])
pred_log = meta_model.predict(meta_test_X)
pred = np.expm1(pred_log)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": pred})
submission.to_csv(r"D:\Data Analytics\Kaggle\submission.csv", index=False)
submission.head()